# Co-Routinen, Threads & Subprozesse

## Concurrent Programming

Das Konzept ***"concurrent programming"*** ist eine Sammlung von Techniken,
um innerhalb eines Programmes mehrere Aufgaben unabhängig -- oder fast unabhängig -- voneinander bearbeiten zu können.
Auf entsprechenden Maschinen, bzw. in einem Verbund von Computern, können diese Aufgaben wenn möglich auch gleichzeitig ausgeführt werden - das nennt sich ***"parallel computing"***.

* [Concurrent Computing](https://en.wikipedia.org/wiki/Concurrent_computing)
* [Parallel Computing](https://en.wikipedia.org/wiki/Parallel_computing)

## Generatoren

Eine der grundlegendsten Formen von concurrency in Python sind **Generatoren**.
Ein Generator verhält sich ähnlich wie eine Funktion, gibt aber eine Folge von Werten zurück,
anstatt nur einen einzigen.

Das Schlüsselwort `yield` markiert die Stelle, an der ein Zwischenergebnis zurückgegeben wird.
Beim nächsten Aufruf von `next()` läuft die Funktion von genau dieser Stelle weiter.

Ein Generator ist ein spezieller [Iterator](https://docs.python.org/3/glossary.html#term-iterator)
und kann in `for`-Schleifen direkt verwendet werden.
`StopIteration` signalisiert das Ende des Generators.

Einfache Beispiele:

In [3]:
def coroutine1():
    yield "abc"
    yield "xyz"
    yield 777

In [4]:
c1 = coroutine1()
print(next(c1))
print(next(c1))
print(next(c1))
print(next(c1))

abc
xyz
777


StopIteration: 

In [5]:
def even_numbers(start_value = 0):
    x = start_value
    while True:
        x += 1
        if x % 2 == 0:
            yield x

In [6]:
en = even_numbers(21)
for i in range(5):
    print(next(en))

22
24
26
28
30


Da Co-Routinen eine spezielle Form von Iteratoren sind,
brechen sie in diesem Fall hier nicht von selbst ab.
Wir müssen daher in dieser `for i in en2` Schleife sicherstellen,
dass irgendwann `break` aufgerufen wird!

In [7]:
en2 = even_numbers(1)
for i in en2:
    print(i)
    if i > 10:
        break

2
4
6
8
10
12


`variable = yield` und `coroutine_instance.send(arg)` erlauben,
Objekte an eine Co-Routine zu schicken.
Achtung, das Timing von `next` und `send` darf nicht aus dem Takt kommen!

In [ ]:
def triple():
    while True:
        print("    Bereit, Werte zu empfangen")
        y = yield
        print(f"    Empfangen: {y}")
        yield y * 3

In [ ]:
t = triple()
next(t)     # Initialisierung des Generators (bis zum ersten yield)
print("init fertig")
z = t.send(21)
print(f"habe {z} erhalten")
next(t)
z = t.send(-4)
print(f"habe {z} erhalten")

**Bonus**: Verkettungen

In [10]:
def oddvals():
    x = 1
    while True:
        yield x
        x += 2

def sumall(co_other):
    sum = 0
    while True:
        sum += next(co_other)
        yield sum

a1 = oddvals()
sa = sumall(a1)
for i in range(10):
    print(next(sa))

1
4
9
16
25
36
49
64
81
100


**Kontrolle:**

In [11]:
[sum(range(1, 2 * x, 2)) for x in range(1, 11)]

[1, 4, 9, 16, 25, 36, 49, 64, 81, 100]

## Threading

Threads ermöglichen echte Parallelausführung innerhalb eines Prozesses.
In Python unterliegen Threads dem [Global Interpreter Lock (GIL)](https://docs.python.org/3/glossary.html#term-global-interpreter-lock),
was bedeutet, dass zu einem Zeitpunkt nur ein Thread Python-Code ausführen kann.
Threads sind daher vor allem bei I/O-gebundenen Aufgaben nützlich (Netzwerk, Dateisystem),
nicht bei rechenintensiven Aufgaben (dafür eignet sich `multiprocessing` besser).

In [13]:
from threading import Thread
from queue import Queue

In [14]:
def calculate(name, input_queue, return_queue):
    sum = 0
    while True:
        c = input_queue.get()
        print("working in " + name)
        if c < 0:
            break
        sum += c
    return_queue.put(sum)
    
input_queue = Queue()
return_queue = Queue()

t1 = Thread(target=calculate, args=("T1", input_queue, return_queue))
t2 = Thread(target=calculate, args=("T2", input_queue, return_queue))

t1.start()
t2.start()

for k in range(99999, 99999+20):
    input_queue.put(k)

input_queue.put(-1)
input_queue.put(-1)

working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T1
working in T2


## Async/Await (moderne Co-Routinen)

Seit Python 3.5 gibt es mit `async def` und `await` eine elegante Syntax für echte Co-Routinen.
Das Modul [`asyncio`](https://docs.python.org/3/library/asyncio.html) stellt die dazugehörige Event-Loop bereit.

**Konzept:**
- `async def` definiert eine Co-Routine (ähnlich einer Funktion, aber unterbrechbar)
- `await` gibt die Kontrolle an die Event-Loop ab, bis das erwartete Ergebnis bereit ist
- Die Event-Loop wechselt effizient zwischen Co-Routinen, wenn diese auf I/O warten

Das ist besonders nützlich bei Netzwerkanfragen, Datenbankzugriffen oder anderen I/O-gebundenen Operationen,
da andere Co-Routinen weiterarbeiten können, während eine auf eine Antwort wartet.

Beobachtung: Obwohl `A` zuerst gestartet wird, endet `B` (mit kürzester Wartezeit) zuerst —
die Co-Routinen laufen *nebenläufig*, nicht sequenziell.

**Vergleich der Ansätze:**

| Methode | Geeignet für | Bemerkung |
|---|---|---|
| `yield`-Generatoren | Datenströme, lazy evaluation | kein echter Concurrency |
| `threading` | I/O-bound, einfache Parallelarbeit | GIL limitiert CPU-Parallelismus |
| `asyncio` (async/await) | viele gleichzeitige I/O-Operationen | modern, effizient, kein GIL-Problem |
| `multiprocessing` | CPU-bound (rechenintensiv) | echte Parallelität, separater Speicher |

In [ ]:
import asyncio

async def aufgabe(name, dauer):
    print(f"  {name}: starte (warte {dauer}s)")
    await asyncio.sleep(dauer)   # gibt Kontrolle ab, simuliert I/O-Wartezeit
    print(f"  {name}: fertig")
    return f"Ergebnis von {name}"

async def main():
    # Alle drei Aufgaben gleichzeitig starten
    ergebnisse = await asyncio.gather(
        aufgabe("A", 0.3),
        aufgabe("B", 0.1),
        aufgabe("C", 0.2),
    )
    for e in ergebnisse:
        print(e)

# In Jupyter/IPython direkt mit await aufrufbar (ab IPython 7):
await main()